# Private B1/B2 benchmark EN/VI — preparation only
Review source: scripts/27_kaggle_b12_benchmark.py. Campaign: docs/planning/kaggle-b12-benchmark-campaign-2026-10-18.md.
Create a **private** notebook only after separate authorization. Attach private dataset `aohkvd/nl2sql-setup-inputs` and private cache output `aohkvd/nl2sql-environment-cache` **version 5**. Confirm saved attachment version in UI; paths are discovered, content is hash checked. Use T4 hardware as in smoke v2. Internet is needed only for exact runtime installation; model inference is offline and needs no HF_TOKEN.
This source performs no reads/writes/install/network/inference by default. No Kaggle API, credentials, uploads, BigQuery, training or model downloads. Accepted notebook 14 remains unchanged inside the immutable bundle.


In [ ]:
ALLOW_PREPARATION = False
ALLOW_ENV_INSTALL = False
ALLOW_REAL_INFERENCE = False
CAMPAIGN_ID = None  # original campaign; approved replacement uses parent-replacement01
SESSION_ID = None  # fresh identity per session; retain all previous outputs
BASELINE = "b1"
LANGUAGE = "en"
REPLICATE = 1
PRIOR_ATTEMPTS = None  # [] only for first B1 EN r01; then carry campaign-ledger.json IDs
PRIOR_COMPLETENESS_REVIEWED = False  # runtime/artifact review only, never test-score tuning
REPLACEMENT_OF = None  # separately approved B2 EN r03 only; retain all 12 original attempted IDs


One full 100-case process per session. Complete matrix: B1/B2 × EN/VI × r01/r02/r03 = 12; N=100 separately per language, never pooled. First: B1 EN r01, all 100 cases. Review only runtime/artifact completeness before later sessions. Preserve failed attempts, never splice/retry invalid or unsafe SQL. Latency thresholds and all research settings stay fixed.

Before real execution: settle campaign/session identities, fresh quota and charging units, private attachments/version, caps/budget and explicit authorization. Set PRIOR_ATTEMPTS from the retained ledger (including failures); archive prior evidence before starting a fresh session. The notebook cannot detect outputs abandoned in another Kaggle session without that ledger. Any attempted ID is refused. Keep a replacement full-process attempt under a separately approved new identity.

For input-only verification enable ALLOW_PREPARATION. Installation additionally requires ALLOW_ENV_INSTALL. With inference disabled the installed runtime performs data/cache/runtime preflight and planning only. Genuine execution requires all three switches. Process cap 5400s; setup/cache cap 1620s; session driver cap 7200s including 120s cleanup reserve. Kaggle's platform hard stop can interrupt cleanup/final manifests; streamed logs survive only if platform saves outputs. Export all outputs after each session.

For a separately authorized full B2 EN r03 replacement only: use CAMPAIGN_ID=<parent>-replacement01 and REPLACEMENT_OF=<parent>-b2-en-r03. Carry all twelve original attempted IDs and review their evidence. The new native ID is <parent>-replacement01-b2-en-r03; the parent failure stays in the ledger. No automatic retry or additional matrix is authorized by this setting.


In [ ]:
DRIVER_SHA256 = '57eacba9b110b82f7a09c384794181b2a8f11222da69f9ec487c9dfe6ca38b9a'
DRIVER_SOURCE = '"""Private notebook operations; unchanged immutable CLI25/CLI17 perform inference.\n\nNo execution on import. Default notebook settings authorize no side effects.\nThis file is embedded verbatim in notebook 16; it is outside the accepted bundle.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport time\nfrom pathlib import Path, PurePosixPath\n\nBUNDLE_SHA256 = "2aa4dac0cb4110783279c3361b4c7618ce601402ed3a40cef168adad75caa6de"\nMODEL_REVISION = "8afb486c1db24fe5011ec46dfbe5b5dccdb575c2"\nENCODER_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"\nINVENTORY_SHA256 = "ae3b7dc2361f151bcc59d68f4b2ddcdef74b99a5fe8374b041190bded84d84f1"\nSETUP_HASHES = {\n    "scripts/26_kaggle_environment.py": (\n        "91be7869ae7223b93e7ff6c3c7a58589b241161ef7a782351f02e105e86d9218"\n    ),\n    "configs/kaggle-b12-model-pins.json": (\n        "6f72c03fe03c379b1ed0844192ef17c94cb60e908c5269f0e49750a32f04998a"\n    ),\n    "configs/kaggle-b12-requirements.lock.txt": (\n        "da3c09c04886fca688aa6a2e54aa1d7c46726be5a4c67b19bc2ba8b9ab36a5cd"\n    ),\n    "configs/kaggle-b12-bootstrap.lock.txt": (\n        "021d7fd19573db49d3acd74056047912457790f20ceffb4d1196ebffa067a9c8"\n    ),\n}\nPROCESS_CAP = 5400\nSESSION_CAP = 7200\nWORKER_OVERHEAD = 60\nCLEANUP_RESERVE = 120\nSETUP_CAP = SESSION_CAP - (PROCESS_CAP + WORKER_OVERHEAD) - CLEANUP_RESERVE\n\n\ndef digest(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef write_new(path, value):\n    with path.open("x") as stream:\n        json.dump(value, stream, ensure_ascii=False, sort_keys=True, indent=2)\n        stream.write("\\n")\n\n\ndef discover_inputs(mount):\n    archives = list(mount.rglob("input-bundle.bundle"))\n    inventories = list(mount.rglob("b12-cache-inventory.json"))\n    if len(archives) != 1 or len(inventories) != 1:\n        raise ValueError("Exactly one accepted bundle and cache inventory required")\n    archive, inventory = archives[0], inventories[0]\n    kit, home = archive.parent, inventory.parent / "hf-home"\n    for name, expected in SETUP_HASHES.items():\n        path = kit / name\n        if not path.resolve().is_relative_to(mount.resolve()) or digest(path) != expected:\n            raise ValueError("Setup fingerprint mismatch: " + name)\n    if digest(archive) != BUNDLE_SHA256 or digest(inventory) != INVENTORY_SHA256:\n        raise ValueError("Accepted archive/cache-v5 inventory fingerprint mismatch")\n    if not home.is_dir() or not home.resolve().is_relative_to(mount.resolve()):\n        raise ValueError("Cache must stay mounted read-only under inputs")\n    return archive, kit, inventory, home\n\n\ndef extract_inputs(archive, workspace):\n    if digest(archive) != BUNDLE_SHA256:\n        raise ValueError("Accepted bundle fingerprint mismatch")\n    if workspace.exists() or workspace.is_symlink():\n        raise ValueError("existing extraction destination")\n    with tarfile.open(archive) as bundle:\n        members = bundle.getmembers()\n        if (\n            len(members) != 182\n            or len({m.name for m in members}) != 182\n            or sum(m.size for m in members) > 128 * 1024 * 1024\n        ):\n            raise ValueError("Bundle member count/size invalid")\n        for m in members:\n            p = PurePosixPath(m.name)\n            if (\n                not m.isfile()\n                or p.is_absolute()\n                or str(p) != m.name\n                or any(x in {".", ".."} or x.startswith(".") for x in m.name.split("/"))\n            ):\n                raise ValueError("Unsafe bundle entry")\n        bundle.extractall(workspace, filter="data")\n\n\ndef runtime_config(root, campaign, home, language):\n    config = json.loads((root / "configs/kaggle-b12-run.template.json").read_bytes())\n    config.update(\n        campaign_id=campaign,\n        language=language,\n        hf_home=str(home),\n        model_revision=MODEL_REVISION,\n        encoder_revision=ENCODER_REVISION,\n        run_timeout_seconds=PROCESS_CAP,\n    )\n    return config\n\n\ndef campaign_plan(root, campaign, home, output):\n    from nl2sparql.kaggle.runner import plan_runs\n\n    return [\n        dict(item, config=config)\n        for language in ("en", "vi")\n        for config in [runtime_config(root, campaign, home, language)]\n        for item in plan_runs(root, config, output)\n    ]\n\n\ndef select_process(plan, baseline, language, replicate, prior_attempts=()):\n    matches = [\n        r\n        for r in plan\n        if (r["baseline"], r["language"], r["replicate"]) == (baseline, language, replicate)\n    ]\n    if len(matches) != 1:\n        raise ValueError("Select exactly one planned process")\n    item = matches[0]\n    if item["run_id"] in prior_attempts:\n        raise ValueError("existing campaign attempt; never resume or overwrite")\n    if Path(item["folder"]).exists() or Path(item["folder"]).is_symlink():\n        raise ValueError("existing run output must not be overwritten")\n    return item\n\n\ndef validate_attempts(settings):\n    """Validate operational identities before writes; retain failed IDs permanently."""\n    for key in ("campaign_id", "session_id"):\n        if not isinstance(settings[key], str) or not re.fullmatch(\n            r"[A-Za-z0-9][A-Za-z0-9_-]{0,59}", settings[key]\n        ):\n            raise ValueError("Set safe explicit " + key)\n    if (\n        settings["baseline"] not in ("b1", "b2")\n        or settings["language"] not in ("en", "vi")\n        or type(settings["replicate"]) is not int\n        or settings["replicate"] not in (1, 2, 3)\n    ):\n        raise ValueError("Choose one baseline/language/replicate")\n    prior = settings["prior_attempts"]\n    provenance = {}\n    parent = settings["campaign_id"]\n    replacement = settings.get("replacement_of")\n    if replacement is not None:\n        # One explicitly selected full B2 EN r03 replacement after the original\n        # twelve attempts. A new native campaign namespace avoids any ID reuse\n        # without modifying the accepted native runner or research configuration.\n        if not isinstance(replacement, str):\n            raise ValueError("Replacement identity must be a string")\n        match = re.fullmatch(r"([A-Za-z0-9][A-Za-z0-9_-]{0,59})-b2-en-r03", replacement)\n        if (\n            match is None\n            or settings["campaign_id"] != match[1] + "-replacement01"\n            or (settings["baseline"], settings["language"], settings["replicate"])\n            != ("b2", "en", 3)\n        ):\n            raise ValueError(\n                "Replacement requires a distinct parent-replacement01 campaign and B2 EN r03"\n            )\n        parent = match[1]\n        provenance = {"parent_campaign_id": parent, "replacement_of": replacement}\n    valid_ids = {\n        f"{parent}-{b}-{lang}-r{rep:02d}"\n        for b in ("b1", "b2")\n        for lang in ("en", "vi")\n        for rep in (1, 2, 3)\n    }\n    selected = (\n        f"{settings[\'campaign_id\']}-{settings[\'baseline\']}-"\n        f"{settings[\'language\']}-r{settings[\'replicate\']:02d}"\n    )\n    if (\n        not isinstance(prior, list)\n        or any(x not in valid_ids for x in prior)\n        or len(set(prior)) != len(prior)\n        or selected in prior\n    ):\n        raise ValueError("Supply prior attempted IDs from retained campaign ledger; no reuse")\n    if replacement is not None and set(prior) != valid_ids:\n        raise ValueError("Review and retain all twelve original attempts before replacement")\n    if not prior and (settings["baseline"], settings["language"], settings["replicate"]) != (\n        "b1",\n        "en",\n        1,\n    ):\n        raise ValueError("First process must be full B1 EN r01; review completeness before more")\n    if prior and settings.get("prior_completeness_reviewed") is not True:\n        raise ValueError("Review prior runtime/artifact completeness before continuing")\n    return provenance\n\n\ndef run_logged(argv, env, log, timeout):\n    """Kill the complete process group on timeout/interrupt, keeping streamed logs."""\n    if timeout <= 0:\n        raise TimeoutError("Session/setup budget exhausted")\n    with log.open("x") as stream:\n        process = subprocess.Popen(\n            argv, env=env, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True\n        )\n        try:\n            code = process.wait(timeout=timeout)\n        except BaseException:\n            # Native CLI17 is a grandchild when CLI25 is the immediate child.\n            # Kill the group even if its leader exited while handling our signal.\n            try:\n                os.killpg(process.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n            process.wait()\n            raise\n    if code:\n        raise RuntimeError(f"Stage failed: exit={code}; inspect {log.name}")\n\n\ndef run_timed_stage(\n    argv,\n    env,\n    log,\n    *,\n    timings,\n    stage,\n    driver_started,\n    budget,\n    deadline_seconds,\n    max_seconds=None,\n):\n    """Persist stage timing even on failure; never serialize commands or environment."""\n    started = time.monotonic()\n    remaining = max(0.0, deadline_seconds - (started - driver_started))\n    timeout = remaining if max_seconds is None else min(max_seconds, remaining)\n\n    def record(event, now, **extra):\n        with timings.open("a") as stream:\n            stream.write(\n                json.dumps(\n                    {\n                        "event": event,\n                        "stage": stage,\n                        "budget": budget,\n                        "driver_elapsed_seconds": now - driver_started,\n                        "elapsed_seconds": now - started,\n                        "timeout_seconds": timeout,\n                        "budget_remaining_seconds": max(\n                            0.0, deadline_seconds - (now - driver_started)\n                        ),\n                        **extra,\n                    },\n                    sort_keys=True,\n                )\n                + "\\n"\n            )\n\n    record("started", started)\n    outcome, error = "completed", {}\n    try:\n        run_logged(argv, env, log, timeout)\n    except BaseException as exc:\n        outcome = (\n            "timed_out" if isinstance(exc, (TimeoutError, subprocess.TimeoutExpired)) else "failed"\n        )\n        error = {"error_type": type(exc).__name__}\n        raise\n    finally:\n        record("finished", time.monotonic(), outcome=outcome, **error)\n\n\ndef cleanup_scratch(scratch, working):\n    if (\n        scratch.is_symlink()\n        or scratch.parent.resolve() != working.resolve()\n        or scratch == working\n        or not scratch.name.startswith("b12-scratch-")\n    ):\n        raise ValueError("Cleanup requires a notebook-owned scratch child")\n    if scratch.exists():\n        shutil.rmtree(scratch)  # shutil does not follow directory symlinks.\n\n\ndef check_complete(root, item):\n    """Runtime/artifact completeness only. Do not gate on accuracy or SQL validity."""\n    from nl2sparql.models.b12.evaluate import load_evaluation_cases\n\n    language = item["language"]\n    snapshot = root / (\n        "data/dataset/test/test-100.jsonl"\n        if language == "en"\n        else "data/dataset/test/test-100-vi.jsonl"\n    )\n    cases = load_evaluation_cases(snapshot, language=language, source_root=root)\n    expected = [case.case_id for case in cases]\n    folder = Path(item["folder"])\n    run = json.loads((folder / "run.json").read_bytes())\n    report = json.loads((folder / "native-report.json").read_bytes())\n    rows = [json.loads(line) for line in (folder / "predictions.jsonl").read_text().splitlines()]\n    logs = [json.loads(line) for line in (folder / "native-log.jsonl").read_text().splitlines()]\n    if (\n        len(expected) != 100\n        or [r["case_id"] for r in rows] != expected\n        or [r["case_id"] for r in logs] != expected\n        or report["prediction_count"] != 100\n        or report["metrics"]["total"] != 100\n        or run["status"] != "completed_native_inference"\n    ):\n        raise ValueError("Incomplete native process; preserve entire attempt")\n    for obj in [run, report, *rows, *logs]:\n        if obj["run_id"] != item["run_id"] or obj["language"] != language:\n            raise ValueError("Native run/language mismatch")\n    if any(obj["baseline"] != item["baseline"] for obj in (run, report)):\n        raise ValueError("Native baseline mismatch")\n    if set(run["native_outputs"]) != {\n        "predictions.jsonl",\n        "native-log.jsonl",\n        "native-report.json",\n    }:\n        raise ValueError("Native output hash inventory incomplete")\n    for row, log in zip(rows, logs, strict=True):\n        prediction = row["prediction"]\n        completion = prediction["completion"]\n        if (\n            prediction["baseline"] != item["baseline"]\n            or completion["synthetic_backend"] is not False\n            or log["synthetic_backend"] is not False\n            or completion["model_revision"] != MODEL_REVISION\n        ):\n            raise ValueError("Native runtime provenance mismatch")\n    for name, expected_hash in run["native_outputs"].items():\n        if digest(folder / name) != expected_hash:\n            raise ValueError("Native artifact digest mismatch")\n    return {\n        "status": "complete_100_native_cases",\n        "language": language,\n        "N": 100,\n        "scientific_ready": False,\n        "accuracy_assessed_by_driver": False,\n    }\n\n\ndef worker(root, home, output, settings, evidence):\n    # The runtime\'s PYTHONPATH points exclusively at immutable bundle source.\n    if sys.version_info[:3] != (3, 11, 15):\n        raise ValueError("Exact smoke Python 3.11.15 required")\n    from nl2sparql.kaggle.bundle import preflight_data\n    from nl2sparql.kaggle.runner import preflight_runtime, run_campaign\n\n    provenance = validate_attempts(settings)\n    plan = campaign_plan(root, settings["campaign_id"], home, output)\n    write_new(evidence / "campaign-plan.json", plan)\n    item = select_process(\n        plan,\n        settings["baseline"],\n        settings["language"],\n        settings["replicate"],\n        settings["prior_attempts"],\n    )\n    config = item["config"]\n    write_new(evidence / "run-config.json", config)\n    data = preflight_data(root)\n    runtime = preflight_runtime(root, config, output)\n    write_new(evidence / "data-preflight.json", data)\n    write_new(evidence / "runtime-preflight.json", runtime)\n    if data["status"] != "data_ready" or runtime["status"] != "ready":\n        raise ValueError("Data/runtime preflight blocked")\n    if not settings["allow_real_inference"]:\n        return "prepared_without_inference"\n    # Reserve identity before any native process. Carry this ledger to next session.\n    write_new(\n        evidence / "campaign-ledger.json",\n        {\n            "campaign_id": settings["campaign_id"],\n            **provenance,\n            "attempted_run_ids": [*settings["prior_attempts"], item["run_id"]],\n            "scientific_ready": False,\n        },\n    )\n    run_campaign(\n        root,\n        config,\n        output,\n        real_inference=True,\n        baseline=item["baseline"],\n        replicate=item["replicate"],\n    )\n    write_new(evidence / "completeness.json", check_complete(root, item))\n    return "completed_native_process"\n\n\ndef prepare(\n    settings, *, mount=Path("/kaggle/input"), working=Path("/kaggle/working"), driver_source=None\n):\n    if not settings["allow_preparation"]:\n        print("BLOCKED: preparation/install/inference disabled")\n        return\n    if settings["allow_real_inference"] and not settings["allow_env_install"]:\n        raise ValueError("Explicit install opt-in required for a fresh pinned runtime")\n    validate_attempts(settings)\n    base = working / ("b12-benchmark-" + settings["campaign_id"])\n    if base.is_symlink() or working.is_symlink():\n        raise ValueError("Unsafe working output alias")\n    base.mkdir(exist_ok=True)\n    evidence = base / ("session-" + settings["session_id"])\n    evidence.mkdir(exist_ok=False)\n    scratch = Path(tempfile.mkdtemp(prefix="b12-scratch-", dir=working))\n    started = time.monotonic()\n    summary = {\n        "status": "failed",\n        "scientific_ready": False,\n        "settings": settings,\n        "session_cap_seconds": SESSION_CAP,\n        "process_cap_seconds": PROCESS_CAP,\n        "setup_cap_seconds": SETUP_CAP,\n        "worker_max_seconds": PROCESS_CAP + WORKER_OVERHEAD,\n        "cleanup_reserve_seconds": CLEANUP_RESERVE,\n    }\n    timings = evidence / "stage-timings.jsonl"\n    stage = "input_discovery"\n    try:\n        timings.touch(exist_ok=False)\n        archive, kit, inventory, home = discover_inputs(mount)\n        workspace, runtime = scratch / "input", scratch / "runtime"\n        bootstrap = scratch / "bootstrap"\n        extract_inputs(archive, workspace)\n        write_new(\n            evidence / "input-provenance.json",\n            {\n                "bundle_sha256": BUNDLE_SHA256,\n                "payload_files": 181,\n                "tar_members": 182,\n                "setup_hashes": SETUP_HASHES,\n                "cache_inventory_sha256": digest(inventory),\n                "cache_version": 5,\n                "hf_home": str(home),\n                "weights_copied": False,\n            },\n        )\n        if not settings["allow_env_install"]:\n            summary["status"] = "inputs_verified_install_disabled"\n            return\n        env = {\n            k: os.environ[k]\n            for k in ("PATH", "LD_LIBRARY_PATH", "CUDA_VISIBLE_DEVICES", "LANG", "LC_ALL")\n            if k in os.environ\n        }\n        env.update(\n            UV_CACHE_DIR=str(scratch / "uv-cache"),\n            UV_PYTHON_INSTALL_DIR=str(scratch / "python"),\n            HF_HUB_DISABLE_IMPLICIT_TOKEN="1",\n            HF_HUB_DISABLE_TELEMETRY="1",\n            PYTHONDONTWRITEBYTECODE="1",\n        )\n\n        def setup(argv, name, custom_env=None):\n            nonlocal stage\n            stage = name\n            run_timed_stage(\n                argv,\n                custom_env or env,\n                evidence / (name + ".log"),\n                timings=timings,\n                stage=name,\n                driver_started=started,\n                budget="setup",\n                deadline_seconds=SETUP_CAP,\n            )\n\n        setup(\n            [\n                sys.executable,\n                "-m",\n                "pip",\n                "--isolated",\n                "install",\n                "--no-deps",\n                "--require-hashes",\n                "--target",\n                str(bootstrap),\n                "-r",\n                str(kit / "configs/kaggle-b12-bootstrap.lock.txt"),\n            ],\n            "install_bootstrap",\n        )\n        uv = str(bootstrap / "bin/uv")\n        setup(\n            [uv, "--no-config", "venv", "--python", "3.11.15", "--managed-python", str(runtime)],\n            "create_python311",\n        )\n        python = str(runtime / "bin/python")\n        setup(\n            [\n                uv,\n                "--no-config",\n                "--no-cache",\n                "pip",\n                "sync",\n                "--python",\n                python,\n                "--require-hashes",\n                "--index-url",\n                "https://pypi.org/simple",\n                str(kit / "configs/kaggle-b12-requirements.lock.txt"),\n            ],\n            "install_locked_environment",\n        )\n        offline = dict(\n            env,\n            PYTHONPATH=str(workspace / "src"),\n            HF_HOME=str(home),\n            HF_HUB_CACHE=str(home / "hub"),\n            HF_HUB_OFFLINE="1",\n            TRANSFORMERS_OFFLINE="1",\n            TOKENIZERS_PARALLELISM="false",\n        )\n        setup(\n            [\n                python,\n                str(kit / "scripts/26_kaggle_environment.py"),\n                "verify-cache",\n                "--pins",\n                str(kit / "configs/kaggle-b12-model-pins.json"),\n                "--hf-home",\n                str(home),\n                "--component",\n                "all",\n            ],\n            "verify_mounted_cache",\n            offline,\n        )\n        # Retain the exact operational source and settings, outside immutable bundle.\n        source = evidence / "benchmark-driver.py"\n        source.write_text(\n            driver_source if driver_source is not None else Path(__file__).read_text()\n        )\n        write_new(evidence / "settings.json", settings)\n        stage = "native_cli25"\n        run_timed_stage(\n            [\n                python,\n                str(source),\n                "--worker",\n                "--root",\n                str(workspace),\n                "--home",\n                str(home),\n                "--output",\n                str(base / "runs"),\n                "--settings",\n                str(evidence / "settings.json"),\n                "--evidence",\n                str(evidence),\n            ],\n            offline,\n            evidence / "worker.log",\n            timings=timings,\n            stage=stage,\n            driver_started=started,\n            budget="session",\n            deadline_seconds=SESSION_CAP - CLEANUP_RESERVE,\n            max_seconds=PROCESS_CAP + WORKER_OVERHEAD,\n        )\n        summary["status"] = (\n            "completed_native_process"\n            if settings["allow_real_inference"]\n            else "prepared_without_inference"\n        )\n    except BaseException as exc:\n        summary.update(error_type=type(exc).__name__, error=str(exc))\n        raise\n    finally:\n        summary.update(stage=stage, wall_seconds=time.monotonic() - started)\n        try:\n            cleanup_scratch(scratch, working)\n            summary["scratch_removed"] = True\n        finally:\n            write_new(evidence / "driver-summary.json", summary)\n            files = {\n                str(p.relative_to(base)): {"bytes": p.stat().st_size, "sha256": digest(p)}\n                for p in base.rglob("*")\n                if p.is_file() and not p.is_symlink()\n            }\n            write_new(evidence / "artifacts-manifest.json", files)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--worker", action="store_true")\n    for name in ("root", "home", "output", "settings", "evidence"):\n        parser.add_argument("--" + name, type=Path)\n    args = parser.parse_args()\n    if not args.worker or any(\n        getattr(args, n) is None for n in ("root", "home", "output", "settings", "evidence")\n    ):\n        parser.error("Operational worker requires all paths; use disabled notebook for preparation")\n    settings = json.loads(args.settings.read_bytes())\n    print(worker(args.root, args.home, args.output, settings, args.evidence), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


In [ ]:
if not ALLOW_PREPARATION:
    print("BLOCKED: preparation/install/inference disabled")
else:
    import hashlib
    if hashlib.sha256(DRIVER_SOURCE.encode()).hexdigest() != DRIVER_SHA256:
        raise ValueError("Operational source fingerprint mismatch")
    scope = {"__name__": "benchmark_notebook"}
    exec(compile(DRIVER_SOURCE, "benchmark-driver", "exec"), scope)
    scope["prepare"]({
        "allow_preparation": ALLOW_PREPARATION,
        "allow_env_install": ALLOW_ENV_INSTALL,
        "allow_real_inference": ALLOW_REAL_INFERENCE,
        "campaign_id": CAMPAIGN_ID, "session_id": SESSION_ID,
        "baseline": BASELINE, "language": LANGUAGE, "replicate": REPLICATE,
        "prior_attempts": PRIOR_ATTEMPTS,
        "replacement_of": REPLACEMENT_OF,
        "prior_completeness_reviewed": PRIOR_COMPLETENESS_REVIEWED,
    }, driver_source=DRIVER_SOURCE)


Retain `/kaggle/working/b12-benchmark-<campaign>/` in full: per-process native artifacts, run.json, stdout/stderr, language caches, per-session source/settings, provenance, preflights, plan, ledger, worker.log, stage-timings.jsonl, summary and hash manifest. Cleanup removes only a fresh notebook-owned `b12-scratch-*` directory; mounted inputs and campaign artifacts remain. Do not rerun a used session ID. No scientific completion is asserted by the driver; canonical adaptation, privacy/cost evidence and separately budgeted BigQuery execution remain T5.4 work. Existing B0 0/100 is English evidence only.
